# Peterson et al. (2021), eNeuro: behaviour and spectral power, from dandiset 000055

Paper: Peterson, Singh, Wang, Rao and Brunton (2021), "Behavioral and Neural Variability of
Naturalistic Arm Movements", eNeuro 8(3), doi:10.1523/ENEURO.0007-21.2021.

Dataset: DANDI:000055 (AJILE12), version 0.220127.0436 (doi:10.48324/dandi.000055/0.220127.0436).
The paper cites no version of the dandiset. 0.220127.0436 is its only published version, and the
draft has the same 55 assets and the same size, so the published version is pinned.

Claims: `claims/000055/10.1523-eneuro.0007-21.2021.yaml`, written from the version of record
(Europe PMC, PMC8225404), with the legends of the Extended Data figures; the figures themselves
were not available to the readers.

Outside inputs, both from the paper's Figshare project and pinned by DOI:
the projection matrices from electrodes to AAL regions (10.6084/m9.figshare.12114408.v1), the
paper's own output of its projection step, used for every claim on regions; and the metadata of
the curated epoch files (10.6084/m9.figshare.12115728.v1), read for their false-positive flags
only (the ECoG in those files is not read). Two link claims, added by this attempt, test that both
match the dataset.

This notebook tests the claims below. The regression claims of Figure 6 are in
`reproduce_peterson_2021_regression.ipynb`.

| Claim | Here |
|---|---|
| figshare-event-link, figshare-roi-link (link claims of this attempt) | tested |
| subject-days, cohort, fig2-event-counts | tested |
| fig2-features-differ-across-subjects, fig2-reach-shape, fig2-bimanual-skew | tested |
| extfig2-1c-contralateral-larger, extfig2-2-feature-correlations, fig2-days-and-waking | tested |
| fig3-group-lfb-decrease-hfb-increase, fig3-sensorimotor-largest, fig3-lfb-more-widespread | tested |
| fig4-precentral-subject-variability | tested |
| fig5-band-power-differs-across-subjects, table1-subject-band-power-ci, fig5-recording-day-effects | tested |
| extfig5-1-baseline-reduces-day-variability, discussion-day-changes-small | tested |
| fig6b, fig6-r2, fig6c, extfig6-2, fig6d (two claims) | regression notebook |

In [ ]:
import os, time, io
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
from concurrent.futures import ProcessPoolExecutor
import matplotlib
import matplotlib.pyplot as plt
import ajile

T0 = time.time()


def kruskal(groups):
    """Kruskal-Wallis over the groups with at least one value; nan when fewer than two."""
    groups = [np.asarray(g) for g in groups if len(g) > 0]
    if len(groups) < 2:
        return np.nan, np.nan
    return stats.kruskal(*groups)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
# All 12 subjects; AJILE_SUBJECTS restricts them while developing.
SUBJECTS = [int(s) for s in os.environ.get("AJILE_SUBJECTS", ",".join(map(str, range(1, 13)))).split(",")]
A_all = ajile.assets()
A = A_all[A_all.subject.isin(SUBJECTS)].reset_index(drop=True)
with ProcessPoolExecutor(8) as ex:
    meta = dict(zip(A.path, ex.map(ajile.read_meta, A.asset_id)))
fig_files = ajile.figshare_files(ajile.EPOCH_ARTICLE)
fig_names = sorted(n for n in fig_files if int(n[5:7]) in SUBJECTS)
with ProcessPoolExecutor(8) as ex:
    figmeta = dict(zip(fig_names, ex.map(ajile.epoch_metadata, fig_names, [fig_files] * len(fig_names))))
roi_txt, roi_files = ajile.roi_matrices()
print(f"{len(A_all)} assets in the version, {len(A)} read; {len(figmeta)} Figshare epoch files; "
      f"loaded in {time.time() - T0:.0f} s")

## Data check

In [ ]:
print("Assets per subject and the days they hold (from the file names):")
print(A_all.groupby("subject").day.apply(lambda d: sorted(d)).to_string())
print("total assets", len(A_all), "; total size", round(A_all["size"].sum() / 1e9), "GB")
rows = []
for _, a in A.iterrows():
    m = meta[a.path]
    el = m["electrodes"]
    rows.append(dict(subject=a.subject, day=a.day, subject_id=m["subject_attrs"].get("subject_id"),
                     species=m["subject_attrs"].get("species"), side=m["reach_side"],
                     events=len(m["reaches"]), reach_events_equal=bool(np.allclose(m["reach_events"], m["reaches"].start_time, atol=0.002)),
                     channels=m["ecog_shape"][1], good=int(el.good.sum()),
                     ecog_hours=round(m["ecog_shape"][0] / m["ecog_rate"] / 3600, 3), ecog_rate=m["ecog_rate"],
                     ecog_unit=m["ecog_unit"], conversion=m["ecog_conversion"], chunks=m["ecog_chunks"],
                     pose_hours=round(m["pose_shape"][0] / m["pose_rate"] / 3600, 3), pose_rate=m["pose_rate"],
                     pose_unit=m["pose_unit"], session_start=m["session_start"],
                     x_mean=round(el.x.mean(), 1)))
files = pd.DataFrame(rows)
print(files.drop(columns=["chunks"]).to_string())
print("subject attributes held:", sorted({k for m in meta.values() for k in m["subject_attrs"]}))
print("electrode table columns:", list(meta[A.path[0]]["electrodes"].columns))
print("ECoG chunk shapes:", sorted({str(c) for c in files.chunks}))
print("files whose ECoG is shorter than the pose:")
print(files[files.ecog_hours < files.pose_hours - 0.01][["subject", "day", "ecog_hours", "pose_hours", "session_start"]].to_string())

The ECoG is stored with `conversion` 1e-6 to volts, so the stored values are microvolts, and the
spectral power below is in dB of microvolts squared. The pose is labelled "meters" but holds image
pixels (values up to several hundred, and the reaches table gives magnitudes in pixels).

Session start times are stored as 19:00 or 03:xx at offset -05:00. Read on the UTC clock they are
midnight, or the time a recording began on a day that started late. The next cell checks this
reading against the coarse behavioural labels: on the UTC reading sleep falls at night.

In [ ]:
pose_rng = []
for p in A.path[:3]:
    f = ajile.open_nwb(A.set_index("path").loc[p, "asset_id"])
    d = f["processing/behavior/Position/R_Wrist/data"][::3000, :]
    pose_rng.append((p, np.nanmin(d), np.nanmax(d)))
    f.close()
print("R_Wrist pose range, every 3000th frame, first three files:", [(p.split("/")[1][:12], round(a), round(b)) for p, a, b in pose_rng])
sleep = []
for p, m in meta.items():
    st = pd.Timestamp(m["session_start"])
    e = m["epochs"]
    s = e[e.label == "Sleep/rest"]
    mid = (s.start + s.stop) / 2
    for clock, t0 in (("UTC", st.tz_convert("UTC")), ("stated offset", st)):
        h0 = t0.hour + t0.minute / 60 + t0.second / 3600
        sleep += [(clock, (h0 + x / 3600) % 24, d) for x, d in zip(mid, s.stop - s.start)]
sl = pd.DataFrame(sleep, columns=["clock", "hour", "dur"])
tab = sl.groupby(["clock", pd.cut(sl.hour, range(0, 25, 3))], observed=False).dur.sum().unstack(0) / 3600
print("Hours labelled Sleep/rest by clock hour, under the two readings of the session start:")
print(tab.round(0).to_string())
print("epochs labels:", pd.concat([m["epochs"] for m in meta.values()]).label.value_counts().to_dict())

### Link claims: the Figshare inputs against the dataset

figshare-event-link. The Figshare epoch files hold one metadata row per event, with a timestamp
in milliseconds from the start of the file, the reach features in frames, and the paper's
false-positive flag. Records are matched on the timestamp. The link holds when every DANDI event
has exactly one Figshare record with the same timestamp, and the features both hold agree
(magnitude and angle as stored, duration, onset speed and overlap converted from frames at 30
frames/s, bimanual class, bimanual ratio, speech ratio).

In [ ]:
ev = ajile.events_table(A, meta, figmeta)
nd = len(ev)
nf = sum(len(v) for v in figmeta.values())
matched = ev.false_pos.notna()
print(f"DANDI events {nd}, Figshare records {nf}, DANDI events matched on timestamp {matched.sum()} ({matched.mean():.1%})")
dup = sum(v.event_timestamp.duplicated().sum() for v in figmeta.values())
print("duplicated timestamps within a Figshare file:", dup)
agree = {
    "magnitude (px)": np.abs(ev.magnitude - ev.reach_r).max(),
    "angle (deg)": np.abs(ev.Reach_angle_degrees - ev.reach_a).max(),
    "duration (s vs frames/30)": np.abs(ev.duration - ev.reach_duration / 30).max(),
    "onset speed (px/s vs px/frame*30)": np.abs(ev.onset_speed - ev.onset_velocity * 30).max(),
    "overlap (s vs frames/30)": np.abs(ev.bimanual_overlap - ev.other_reach_overlap / 30).max(),
    "speech ratio": np.abs(ev.speech_ratio - ev.audio_ratio).max(),
    "bimanual ratio (where Figshare has one)": np.nanmax(np.abs(ev.bimanual_ratio - ev.I_over_C_ratio)),
}
for k, v in agree.items():
    print(f"  largest difference, {k}: {v:.2e}")
print("  bimanual class differs from Figshare 'bimanual':", int((ev.bimanual_class != ev.bimanual).sum()),
      "; differs from 'other_lead_lag exists':", int((ev.bimanual_class != ev.other_lead_lag.notna()).sum()))
print("  Figshare ratios missing:", int(ev.I_over_C_ratio.isna().sum()), "; DANDI values there:",
      ev.bimanual_ratio[ev.I_over_C_ratio.isna()].unique())
print("  Figshare day equals the DANDI file's day:", bool((ev.day_fig == ev.day).all()) if "day_fig" in ev else None)
print("false positives per subject (Figshare flag):")
fp = ev.groupby("subject").false_pos.agg(events="size", false_pos="sum")
fp["kept"] = fp.events - fp.false_pos
print(fp.astype(int).T.to_string())
print(f"false positives overall: {int(ev.false_pos.sum())} of {nd} ({ev.false_pos.mean():.1%}); the dataset holds no flag for them")

**Result: reproduced.** All 8088 dataset events have exactly one Figshare record with the same
timestamp (100.0%), no timestamp repeats within a file, and the features both hold agree to
within 4.9e-07 (onset speed; 5e-08 or less for the others), with no event whose bimanual class
differs. The Figshare flags mark 1157 events (14.3%) as false positives; the dataset does not
mark them.

figshare-roi-link. Each projection file has a header of 116 AAL regions, a density row, and one
row of weights per electrode. The link holds when each subject's file has one row per channel of
the dataset in the same order. Rows carry no identifier, so the order is checked through
positions: for each electrode, the weighted mean of the region centroids that the Figshare
centroid file gives is compared with the electrode's MNI position from the dataset (mirrored to
the left), against the same distance with the rows of the file shuffled.

In [ ]:
rng = np.random.default_rng(55)
cent_txt = {s: ajile.requests.get(roi_files[f"aal_{s:02d}_ROIcentroids_Lside.csv"]["download_url"], timeout=60).text for s in SUBJECTS}
roi_rows = []
W_all, DENS = {}, {}
for s in SUBJECTS:
    dens, W = ajile.roi_tables(roi_txt[s])
    full = pd.read_csv(io.StringIO(roi_txt[s]))
    cen = pd.read_csv(io.StringIO(cent_txt[s]))
    cen.index = full.columns
    cenL = cen.loc[[c for c in full.columns if c.endswith("_L")]]
    cenL.index = [c[:-2] for c in cenL.index]
    p0 = A.path[A.subject == s].iloc[0]
    el = meta[p0]["electrodes"]
    nch = len(el)
    extra = W.iloc[nch:]
    Wn = W.iloc[:nch].to_numpy(float)
    xyz = el[["x", "y", "z"]].to_numpy(float).copy()
    xyz[:, 0] = -np.abs(xyz[:, 0])
    valid = cenL.abs().sum(axis=1).to_numpy() > 0
    C = cenL.to_numpy(float)

    def dist(Wm):
        w = Wm[:, valid] / Wm[:, valid].sum(1, keepdims=True)
        return np.nanmedian(np.linalg.norm(w @ C[valid] - xyz, axis=1))
    d_true = dist(Wn)
    d_shuf = [dist(Wn[rng.permutation(nch)]) for _ in range(200)]
    same_pos = all(np.allclose(meta[p]["electrodes"][["x", "y", "z"]].to_numpy(), el[["x", "y", "z"]].to_numpy())
                   for p in A.path[A.subject == s])
    roi_rows.append(dict(subject=s, channels=nch, rows=len(W), extra_rows=len(extra),
                         extra_rows_all_zero=bool((extra.to_numpy() == 0).all()) if len(extra) else True,
                         median_dist_mm=round(d_true, 1), shuffled_min=round(min(d_shuf), 1),
                         shuffled_median=round(np.median(d_shuf), 1), positions_same_all_days=same_pos))
    W_all[s] = W.iloc[:nch]
    DENS[s] = dens
roi_link = pd.DataFrame(roi_rows)
print(roi_link.to_string(index=False))
dens_mean = pd.DataFrame(DENS).T.mean()
sel_regions = dens_mean[dens_mean > 3].sort_values(ascending=False)
print("regions with electrode density averaged over subjects above 3:", sel_regions.round(2).to_dict())
print("the claims file's eight regions:", ajile.REGIONS, "; same set:", set(sel_regions.index) == set(ajile.REGIONS))

**Result: reproduced.** Every subject's matrix has one row per channel of the dataset; S01's has
4 more rows, all zero. The median distance between an electrode's position and the weighted mean
of its region centroids is 15.6 to 27.6 mm, below the smallest of 200 row-shuffled distances in
every subject (44.9 to 54.9 mm). The eight regions with density averaged over subjects above 3
are the claims file's eight.

### Clock of the ECoG

In three files the ECoG is 15.5 to 15.9 h long while the pose and the event times span 24 h, and
some events lie after the end of the ECoG. In each of them the ECoG length plus the session start
(on the UTC clock) is 24 h. This fits an ECoG that starts at the session start while the events and
the pose count from midnight. Below, events of those files are read both ways: the stored clock
(ECoG time = event time) and the shifted one (ECoG time = event time minus the session start). The
movement response (LFB decrease and HFB increase over 0 to 0.5 s against -1.5 to -1 s, averaged
over good electrodes with the precentral projection weights) decides which clock holds.

In [ ]:
proj = {}
for _, a in A.iterrows():
    el = meta[a.path]["electrodes"]
    proj[a.path] = np.nan_to_num(ajile.projection(W_all[a.subject], el.good.to_numpy()))
short = files[files.ecog_hours < files.pose_hours - 0.01]
short_paths = [A.path[(A.subject == r.subject) & (A.day == r.day)].iloc[0] for r in short.itertuples()]
cjobs = []
for p in short_paths:
    e = ev[ev.path == p]
    n = meta[p]["ecog_shape"][0] / ajile.FS
    off = e.offset.iloc[0]
    on = e.start_time.to_numpy()
    ok = (on - off > 6) & (on < n - 6) & (on > 6)        # inside the ECoG on both clocks
    cjobs.append(((p, "stored"), e.asset_id.iloc[0], on[ok][:40], proj[p]))
    cjobs.append(((p, "shifted"), e.asset_id.iloc[0], on[ok][:40] - off, proj[p]))
t = time.time()
cres = ajile.extract_many(cjobs, workers=6)
pre = ajile.REGIONS.index("Precentral")
for p in short_paths:
    for clock in ("stored", "shifted"):
        r = cres[(p, clock)]
        w = proj[p][:, pre]
        lfb = r["band"][:, :, 0] @ w
        hfb = r["band"][:, :, 2] @ w
        print(f"{p.split('/')[1][:12]} {clock:8s} n={len(lfb)} precentral LFB {np.mean(lfb):+.2f} dB "
              f"(Wilcoxon p={stats.wilcoxon(lfb).pvalue:.2g}), HFB {np.mean(hfb):+.2f} dB (p={stats.wilcoxon(hfb).pvalue:.2g})")
    e = ev[ev.path == p]
    n = meta[p]["ecog_shape"][0] / ajile.FS
    print(f"   events outside the ECoG on the stored clock: {int(((e.start_time + 5) > n).sum())} of {len(e)}; "
          f"on the shifted clock: {int(((e.start_time - e.offset - 5) < 0).sum() + ((e.start_time - e.offset + 5) > n).sum())}")
print(f"({time.time() - t:.0f} s)")

The shifted clock is used for these three files from here on (ECoG time = event time minus the
session start on the UTC clock; for the other 52 files the session start is midnight and nothing
changes).

### Spectral measures for every event

One pass over the ECoG of every event: Welch PSD means for the segment rejection, Morlet power
per electrode in the two bands, region spectrograms with the paper's projection weights (bad
electrodes of the file set to 0, each region renormalised), and the clock check of the next cell.

In [ ]:
jobs = []
for p in A.path:
    e = ev[ev.path == p]
    jobs.append((p, e.asset_id.iloc[0], (e.start_time - e.offset).to_numpy(), proj[p]))
t = time.time()
spec = ajile.extract_many(jobs, workers=12)
print(f"spectral pass over {sum(len(j[2]) for j in jobs)} events in {len(jobs)} files: {time.time() - t:.0f} s "
      f"(slowest file {max(r['seconds'] for r in spec.values()):.0f} s)")
TB = next(iter(spec.values()))["tbins"]
F = ajile.FREQS
ev["ok"] = np.concatenate([spec[p]["ok"] for p in A.path])
print("events whose 10 s segment is outside the ECoG:", int((~ev.ok).sum()))

Clock check for every file: per event, the precentral-weighted LFB and HFB change over 0 to 0.5 s
after onset against -1.5 to -1 s, and the same measure around a pseudo-onset 3 s before the
event (window -3 to -2.5 s against -4.5 to -4 s). A shared clock shows a stronger LFB decrease
and HFB increase at the true onset; the per-file test is a Wilcoxon signed-rank test of the
per-event differences (true minus pseudo-onset).

In [ ]:
rows = []
for p in A.path:
    r = spec[p]
    w = proj[p][:, pre]
    ok = r["ok"]
    l, h = r["band"][ok, :, 0] @ w, r["band"][ok, :, 2] @ w
    lc, hc = r["ctrl"][ok, :, 0] @ w, r["ctrl"][ok, :, 1] @ w
    sub = ev[ev.path == p]
    rows.append(dict(subject=sub.subject.iloc[0], day=sub.day.iloc[0], n=int(ok.sum()),
                     lfb_onset=l.mean(), lfb_pseudo=lc.mean(), p_lfb=stats.wilcoxon(l - lc).pvalue,
                     hfb_onset=h.mean(), hfb_pseudo=hc.mean(), p_hfb=stats.wilcoxon(h - hc).pvalue))
clock = pd.DataFrame(rows)
clock["contrast"] = (clock.lfb_onset < clock.lfb_pseudo) & (clock.p_lfb < 0.05) | (clock.hfb_onset > clock.hfb_pseudo) & (clock.p_hfb < 0.05)
print(clock.round(3).to_string(index=False))
print(f"files where the onset differs from the pseudo-onset in the expected direction (p<0.05 in LFB or HFB): "
      f"{int(clock.contrast.sum())} of {len(clock)}")
print("files where it could not be established:", clock[~clock.contrast][["subject", "day"]].values.tolist())

### Event sets

(a) every event in the dataset; (b) after dropping the Figshare false positives and the segments
rejected on their power spectral density. A question on how that PSD is computed was answered
(`questions/000055.yaml`), and the criterion of fig2-event-counts now names the readings: one
PSD per 10 s segment, a segment dropped when its power is below 0 dB or its 115-125 Hz power is
more than 3 SD above the subject's segments, with the 0 dB in microvolts. (b1), the main
reading: the PSD averaged over good electrodes. (b2): the same on each good electrode, a segment
dropped when any good electrode fails. A variant takes the 3 SD over all subjects' segments
instead of the subject's ((b1p) and (b2p); for (b2p) the mean and SD are over every good
electrode of every segment, as electrodes differ between subjects). Another variant, (b3), adds
a time-resolved test: it drops what (b1) drops and also any segment whose Morlet power,
averaged over good electrodes and frequencies, is below 0 dB in a 0.1 s bin from -1.5 s to 1.5
s (this catches ECoG zeroed by the preprocessing inside the analysed window, which the PSD of
the whole 10 s does not). The paper gives neither the estimator nor the frequencies of the 0 dB
test; here they are Welch's method with 1 s windows, in microvolts squared per Hz, and 1-200 Hz.
The false positives alone are also shown.

The claims that use reading (b) are decided on (b1), the main reading; (b2), (b1p), (b2p) and
(b3) are shown beside it where they can be computed.


In [ ]:
rej1, rej2, ev["psd_db"], rej3 = ajile.rejection(ev, A, meta, spec, SUBJECTS)
ev["rej_b1"] = rej1 & ev.ok
ev["rej_b2"] = rej2 & ev.ok
ev["rej_b3"] = rej3 & ev.ok
ev["set_a"] = True
ev["set_fp"] = ev.false_pos == 0
ev["set_b1"] = ev.set_fp & ev.ok & ~ev.rej_b1
ev["set_b2"] = ev.set_fp & ev.ok & ~ev.rej_b2
ev["set_b3"] = ev.set_fp & ev.ok & ~ev.rej_b3
rej1p, rej2p = ajile.rejection_pooled(ev, A, meta, spec, SUBJECTS)
ev["rej_b1p"] = rej1p & ev.ok
ev["rej_b2p"] = rej2p & ev.ok
ev["set_b1p"] = ev.set_fp & ev.ok & ~ev.rej_b1p
ev["set_b2p"] = ev.set_fp & ev.ok & ~ev.rej_b2p
print("PSD averaged over good electrodes, dB (uV^2/Hz), 1-200 Hz: range", ev.psd_db.min().round(1), "to", ev.psd_db.max().round(1))
print(ev.groupby("subject")[["set_a", "set_fp", "set_b1", "set_b2", "set_b1p", "set_b2p", "set_b3", "rej_b1", "rej_b2", "rej_b1p", "rej_b2p", "rej_b3"]].sum().T.to_string())
zb = ev.rej_b3 & ~ev.rej_b1 & ev.set_fp
print(f"segments that (b3) drops beyond (b1), among events not flagged false positive: {int(zb.sum())}")
print(f"events of (b1p) that differ from (b1): {int((ev.set_b1p != ev.set_b1).sum())}; of (b2p) from (b2): {int((ev.set_b2p != ev.set_b2).sum())}")
with np.errstate(divide="ignore"):
    low0 = {s: 0 for s in SUBJECTS}
    for p in A.path:
        s = int(A.subject[A.path == p].iloc[0])
        g = meta[p]["electrodes"].good.to_numpy()
        r = spec[p]
        db = 10 * np.log10(r["psd_all"][:, g])
        low0[s] += int((r["ok"] & (np.nanmin(db, 1) < 0)).sum())
print("(b2): segments with a good electrode below 0 dB over 1-200 Hz, by subject:", low0)


## Claims on subjects, days and electrodes

> subject-days (link): The dataset's subjects and recording days are the paper's subjects S01 to
> S12 and the recording days the paper lists for each.

In [ ]:
table2 = {1: [3, 4, 5, 7], 2: [3, 4, 5, 6], 3: [3, 4, 5, 6], 5: [3, 4, 7]}
table2.update({s: [3, 4, 5, 6, 7] for s in [4, 6, 7, 8, 9, 10, 11, 12]})
have = A_all.groupby("subject").day.apply(lambda d: sorted(d)).to_dict()
for s in range(1, 13):
    print(f"S{s:02d} paper {table2[s]} dataset {have.get(s)} {'same' if have.get(s) == table2[s] else 'DIFFERENT'}")
print("subject-days in the dataset:", len(A_all), "; in Table 2:", sum(len(v) for v in table2.values()))
print("subject ids inside the files equal the file names:", all(int(meta[p]["subject_attrs"]["subject_id"]) == s for p, s in zip(A.path, A.subject)))
print("Figshare file names (subject, day) equal the DANDI ones:",
      sorted((int(n[5:7]), int(n.split("_day_")[1][0])) for n in figmeta) == sorted(zip(A.subject, A.day)))

**Result: reproduced.** The 12 subjects hold exactly the days of Table 2, 55 subject-days, the
subject ids inside the files equal the file names, and the Figshare files carry the same subject
and day pairs.

> cohort: Twelve subjects (8 male, 4 female, 29.4 +- 7.9 years) had on average 87.0 +- 12.9
> surface electrodes; five also had 23.2 +- 12.1 depth electrodes; five were implanted in the
> right hemisphere and seven in the left.

Surface electrodes are those of groups whose description does not say "depth"; depth electrodes
those whose description does. The hemisphere is the side of the mean electrode x coordinate and,
separately, the side of the reaching wrist (the contralateral one).

In [ ]:
crow = []
for s in SUBJECTS:
    p0 = A.path[A.subject == s].iloc[0]
    m = meta[p0]
    el = m["electrodes"]
    desc = el.group_name.map(m["groups"])
    depth = desc.str.contains("depth", case=False)
    crow.append(dict(subject=s, channels=len(el), surface=int((~depth).sum()), depth=int(depth.sum()),
                     depth_groups=",".join(sorted(el.group_name[depth].unique())),
                     hemisphere_x="right" if el.x.mean() > 0 else "left",
                     wrist=m["reach_side"], sex=m["subject_attrs"].get("sex"), age=m["subject_attrs"].get("age")))
coh = pd.DataFrame(crow)
print(coh.to_string(index=False))
print("subjects:", len(coh))
print(f"mean surface electrodes per subject: {coh.surface.mean():.1f} (SD {coh.surface.std():.1f})")
dd = coh[coh.depth > 0]
print(f"subjects with depth electrodes: {len(dd)}; mean depth electrodes among them {dd.depth.mean():.1f} (SD {dd.depth.std():.1f})")
print("right hemisphere by x:", int((coh.hemisphere_x == 'right').sum()), "; by left-wrist events:", int((coh.wrist == 'l_wrist').sum()))
print("sex and age in the files:", coh.sex.unique(), coh.age.unique(), "(not held: not testable)")

**Result: partially reproduced.** The files hold 12 subjects, a mean of 84.8 surface electrodes
(paper 87.0 +- 12.9), and 5 subjects implanted on the right by the mean x coordinate and by the
reaching wrist (paper 5). Counting as depth electrodes the groups whose description says
"depth", 7 subjects have them (paper 5), with a mean of 23.7 among them (paper 23.2 +- 12.1).
Sex and age are not in the files, so the 8 males and the mean age are not testable.

> fig2-event-counts: Each subject contributed between 151 and 947 movement events (median 640
> across subjects).

In [ ]:
cnt = ev.groupby("subject")[["set_a", "set_fp", "set_b1", "set_b2", "set_b1p", "set_b2p", "set_b3"]].sum()
print(cnt.T.to_string())
exp = [(151, 15.1), (947, 94.7), (640, 64)]
for c in cnt.columns:
    v = [cnt[c].min(), cnt[c].max(), cnt[c].median()]
    ok = [abs(x - e) <= tol for x, (e, tol) in zip(v, exp)]
    print(f"reading {c}: fewest {v[0]}, most {v[1]}, median {v[2]}; within tolerance {ok} -> {'all three' if all(ok) else 'no'}")

**Result: reproduced** under reading (b). With the Figshare false positives and the rejected
segments dropped, the fewest, most and median events per subject are 150, 946 and 638.5 under
(b1) and 150, 942 and 627.0 under (b3), all within tolerance; the false positives alone give
151, 947 and 639.5, the paper's numbers. Reading (a), all events in the dataset, gives 211, 996
and 770.5, with the fewest and the median outside tolerance. Rejection reading (b2) leaves no
events in S09, S10 and S12.

## Behaviour (Figure 2 and Extended Data Figures 2-1 and 2-2)

> fig2-features-differ-across-subjects: All ten behavioural and environmental features differ
> significantly across subjects.

Features as the paper gives them: day of recording (as a number), time of day (hour of the
day), reach duration (s), reach magnitude (px), reach angle (folded to -90 to 90 degrees), onset
speed (px/s), speech ratio, bimanual ratio (missing values are 0.5 in the dataset), bimanual
overlap (s) and bimanual class.

In [ ]:
FEAT = {"day": "day", "time of day (h)": "hour", "reach duration (s)": "duration", "reach magnitude (px)": "magnitude",
        "reach angle (deg)": "angle_folded", "onset speed (px/s)": "onset_speed", "speech ratio": "speech_ratio",
        "bimanual ratio": "bimanual_ratio", "bimanual overlap (s)": "bimanual_overlap", "bimanual class": "bimanual_class"}
kw = []
for setname in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    e = ev[ev[setname]]
    for k, c in FEAT.items():
        h, p = kruskal([g[c].to_numpy() for _, g in e.groupby("subject")])
        kw.append(dict(events=setname, feature=k, H=h, p=p))
kw = pd.DataFrame(kw)
print(kw.pivot(index="feature", columns="events", values="p").to_string(float_format=lambda x: f"{x:.2e}"))
for s in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    print(s, ": features with p < 0.001:", int((kw[kw.events == s].p < 0.001).sum()), "of 10")
print("feature ranges (reading b1):")
print(ev[ev.set_b1][list(FEAT.values())].describe().T[["min", "50%", "max"]].round(3).to_string())

**Result: reproduced.** All ten features differ across subjects with p < 0.001 under (b1), (b3)
and (a): the largest p is 1.47e-30 (reach angle) under (b1) and 6.03e-30 under (b3).

> fig2-reach-shape: Reach duration and reach magnitude vary little across subjects, with most
> reaches lasting under 2 s and covering under 200 pixels, and reach angle distributions tend to
> be bimodal with peaks near +90 and -90 degrees, with few side-to-side reaches.

The kernel density uses scipy's Gaussian kernel with Scott's bandwidth (the seaborn default the
claims file names), evaluated at 1 degree steps on -90 to 90. A local maximum is a grid point
higher than its neighbours; an end point of the grid counts when it is higher than its one
neighbour (a standard choice: a peak at the edge of the folded range is a peak at +-90).

In [ ]:
grid = np.arange(-90, 91, 1.0)


def angle_modes(a):
    k = stats.gaussian_kde(a, bw_method="scott")(grid)
    pk = [i for i in range(len(grid)) if (i == 0 or k[i] > k[i - 1]) and (i == len(grid) - 1 or k[i] > k[i + 1])]
    top = sorted(pk, key=lambda i: -k[i])[:2]
    locs = sorted(grid[top])
    bim = len(locs) == 2 and abs(locs[0] + 90) <= 15 and abs(locs[1] - 90) <= 15 and k[90] < min(k[top])
    return locs, k[90], [k[i] for i in top], bim


def shape_table(e):
    rows = []
    for s, g in list(e.groupby("subject")) + [("pooled", e)]:
        locs, k0, kt, bim = angle_modes(g.angle_folded.to_numpy())
        rows.append(dict(subject=s, n=len(g), frac_dur_lt2=(g.duration < 2).mean(), med_dur=g.duration.median(),
                         frac_mag_lt200=(g.magnitude < 200).mean(), med_mag=g.magnitude.median(),
                         peaks=locs, dens0=k0, peak_dens=np.round(kt, 5), bimodal_pm90=bim,
                         frac_side=(g.angle_folded.abs() < 30).mean()))
    return pd.DataFrame(rows)


for setname in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    t = shape_table(ev[ev[setname]])
    print(f"events {setname}")
    print(t.round(3).to_string(index=False))
    subj = t[t.subject != "pooled"]
    pooled = t[t.subject == "pooled"].iloc[0]
    dur_ok = (subj.frac_dur_lt2 > 0.5).sum()
    mag_ok = (subj.frac_mag_lt200 > 0.5).sum()
    print(f"  duration part: pooled {pooled.frac_dur_lt2:.3f}, subjects with >half under 2 s {dur_ok} of {len(subj)}")
    print(f"  magnitude part: pooled {pooled.frac_mag_lt200:.3f}, subjects with >half under 200 px {mag_ok} of {len(subj)}")
    print(f"  angle part: pooled bimodal at +-90 {pooled.bimodal_pm90}; subjects {int(subj.bimodal_pm90.sum())} of {len(subj)}")

**Result: partially reproduced.** More than half of the reaches last under 2 s and cover under
200 pixels in all 12 subjects and pooled (pooled 0.918 and 0.914 under (b1)). The reach angle
part does not hold under (b1), (b3) or (a): the highest density peak is at 62 to 75 degrees in
every subject, the other peak below 0 is at -54 to -73 degrees in nine subjects (S02 and S06
have it near 0 and S04 has one peak), and in 0 of 12 subjects and pooled (-72 and 75 degrees)
are both peaks within 15 degrees of +-90.

> fig2-bimanual-skew: Bimanual ratio and bimanual overlap are skewed toward unimanual movement of
> the contralateral arm, less so for subjects 02, 04 and 05, while the bimanual class is mostly
> bimanual.

In [ ]:
for setname in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    e = ev[ev[setname]]
    rows = []
    for s, g in list(e.groupby("subject")) + [("pooled", e)]:
        rows.append(dict(subject=s, med_ratio=g.bimanual_ratio.median(), skew_overlap=stats.skew(g.bimanual_overlap),
                         frac_bimanual=g.bimanual_class.mean()))
    t = pd.DataFrame(rows)
    subj = t[t.subject != "pooled"].copy()
    pooled = t[t.subject == "pooled"].iloc[0]
    top_ratio = set(subj.nlargest(5, "med_ratio").subject)
    low_skew = set(subj.nsmallest(5, "skew_overlap").subject)
    print(f"events {setname}")
    print(t.round(3).to_string(index=False))
    print(f"  ratio part: pooled median {pooled.med_ratio:.3f}; subjects with median < 0.5: {int((subj.med_ratio < 0.5).sum())} of {len(subj)}")
    print(f"  overlap part: pooled skewness {pooled.skew_overlap:.3f}; subjects with skewness > 0: {int((subj.skew_overlap > 0).sum())} of {len(subj)}")
    print(f"  class part: pooled fraction bimanual {pooled.frac_bimanual:.3f}; subjects above half: {int((subj.frac_bimanual > 0.5).sum())} of {len(subj)}")
    print(f"  five highest median ratios: {sorted(top_ratio)}; five lowest overlap skewness: {sorted(low_skew)}")
    print(f"  S02, S04, S05 among the highest ratios: {[s in top_ratio for s in (2, 4, 5)]}; among the lowest skewness: {[s in low_skew for s in (2, 4, 5)]}")

**Result: blocked (methods underspecified).** The ratio, overlap and class parts hold pooled and
in 12 of 12 subjects under every event set (pooled median ratio 0.115, overlap skewness 1.094,
fraction bimanual 0.717 under (b1)). S02, S04 and S05 are among the five highest median ratios
under every set; among the five lowest overlap skewness S04 is under (b3) and (a) but not under
(b1), where S10 takes its place. The criterion names no event set and the rejection rule is open
(question asked), so the claim is reproduced under (b3) and (a) and partially reproduced under
(b1).

> extfig2-1c-contralateral-larger: Across all 12 subjects, the median displacement of the
> contralateral wrist after movement onset is substantially greater than that of the ipsilateral
> wrist.

Wrist positions are read from the pose at 30 frames/s around each event (pose and events share
the midnight clock in every file, see the data check). Displacement is the Euclidean distance in
pixels from (a) the mean position over -1.5 to -1 s and (b) the position at onset; per subject,
the median over events at each frame, and its peak over 0 to 1.5 s.

In [ ]:
t = time.time()
wjobs = [(p, ev.asset_id[ev.path == p].iloc[0], ev.start_time[ev.path == p].to_numpy(), meta[p]["reach_side"]) for p in A.path]
wr = ajile.wrist_many(wjobs, workers=12)
tw = wr[A.path[0]][1]
print(f"wrist traces read in {time.time() - t:.0f} s; {len(tw)} frames from {tw[0]} to {tw[-1]} s")
base_m = (tw >= -1.5 - 1e-9) & (tw <= -1 + 1e-9)
post_m = (tw >= 0) & (tw <= 1.5 + 1e-9)
i0 = int(np.argmin(np.abs(tw)))
rows = []
for setname in ("set_b1", "set_b1p", "set_b3", "set_a"):
    for s in SUBJECTS:
        disp = {}
        for side in ("contra", "ipsi"):
            arrs = [wr[p][0][side][ev[ev.path == p][setname].to_numpy()] for p in A.path[A.subject == s]]
            X = np.concatenate(arrs)
            for reading, ref in (("a", np.nanmean(X[:, base_m], 1, keepdims=True)), ("b", X[:, i0:i0 + 1])):
                d = np.linalg.norm(X - ref, axis=2)
                disp[(side, reading)] = np.nanmedian(d, 0)
        for reading in ("a", "b"):
            c, i = disp[("contra", reading)][post_m].max(), disp[("ipsi", reading)][post_m].max()
            rows.append(dict(events=setname, reading=reading, subject=s, contra_peak=c, ipsi_peak=i, ratio=c / i, holds=c > i))
wt = pd.DataFrame(rows)
print(wt.pivot_table(index="subject", columns=["events", "reading"], values="ratio").round(2).to_string())
print(wt.groupby(["events", "reading"]).holds.sum().to_string())
print("contralateral and ipsilateral peaks (px), reading b1:")
print(wt[wt.events == "set_b1"].pivot_table(index="subject", columns="reading", values=["contra_peak", "ipsi_peak"]).round(1).to_string())
nan_frac = np.mean([np.isnan(wr[p][0]["contra"]).any(axis=(1, 2)).mean() for p in A.path])
print(f"mean fraction of events with a missing contralateral pose frame in -1.5 to 1.5 s: {nan_frac:.3f} (nan frames are skipped by the medians)")

**Result: reproduced.** The peak of the median contralateral displacement over 0 to 1.5 s
exceeds the ipsilateral one in 12 of 12 subjects under both rest positions and every event set;
the ratio is 1.30 (S04) to 39.8 (S12) under reading (a) with events (b1), and the contralateral
peaks are 55.8 to 157.2 pixels.

> extfig2-2-feature-correlations: Over all events, reach magnitude correlates with reach duration
> (r = 0.26) and onset speed (r = 0.56), reach duration with bimanual overlap (r = 0.48), and the
> bimanual features with one another (overlap and ratio 0.51, class and ratio 0.50, overlap and
> class 0.61).

In [ ]:
pairs = [("magnitude", "duration", 0.26), ("magnitude", "onset_speed", 0.56), ("duration", "bimanual_overlap", 0.48),
         ("bimanual_overlap", "bimanual_ratio", 0.51), ("bimanual_class", "bimanual_ratio", 0.50),
         ("bimanual_overlap", "bimanual_class", 0.61)]
for setname in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    C = ev[ev[setname]][[c for c in FEAT.values()]].corr()
    vals = [C.loc[a, b] for a, b, _ in pairs]
    within = [abs(v - e) <= 0.1 * e for v, (_, _, e) in zip(vals, pairs)]
    print(f"events {setname} (n={int(ev[setname].sum())}): " + "; ".join(f"{a}-{b} r={v:.3f} (paper {e})" for (a, b, e), v in zip(pairs, vals)))
    print(f"   within 10%: {within} -> {sum(within)} of 6")

**Result: reproduced.** Under (b1) the six correlations are 0.259, 0.562, 0.475, 0.508, 0.496
and 0.609 (paper 0.26, 0.56, 0.48, 0.51, 0.50, 0.61), all within 10%; under (b3) also 6 of 6.
With all events (a), 5 of 6 are within 10% (magnitude and duration 0.231).

> fig2-days-and-waking: Reach counts were fairly even across recording days except for subjects
> 04, 05 and 09, each with one day holding most events, and events occurred mostly during waking
> hours.

In [ ]:
for SET in ("set_b1", "set_b2", "set_b1p", "set_b3", "set_a"):
    e = ev[ev[SET]]
    dc = e.groupby(["subject", "day"]).size().unstack()
    busiest = dc.max(axis=1) / dc.sum(axis=1)
    wake = e.groupby("subject").hour.apply(lambda h: ((h >= 8) & (h < 24)).mean())
    out = pd.DataFrame({"busiest_day_frac": busiest, "busiest_day": dc.idxmax(axis=1), "waking_frac": wake})
    print(f"events {SET}")
    print(dc.fillna(0).astype(int).to_string())
    print(out.round(3).to_string())
    print("  subjects whose busiest day holds more than half:", list(out.index[out.busiest_day_frac > 0.5]))
    print("  subjects with more than half of events in 08:00-24:00:", int((out.waking_frac > 0.5).sum()), "of", len(out))

**Result: partially reproduced.** The busiest day holds more than half of the events in S05
(0.628) and S09 (0.553) but not in S04 (0.429, day 7), and in none of the other nine, under
(b1); (b3) and (a) give the same two subjects. More than half of the events fall between 08:00
and 24:00 in 12 of 12 subjects (0.662 to 0.970). Only the waking part holds.

## Spectral power (Figures 3 to 5, Table 1, Extended Data Figure 5-1)

Region power per event is the projection of each electrode's baseline-subtracted Morlet power
(dB) with the paper's weights. The group spectrogram and its mask follow the claims file:
per subject the median over events, the mean over subjects, the group median over -1.5 to -1 s
subtracted again; the null takes, for each frequency, each subject's baseline bins shuffled
and the median over subjects, about 2000 values per frequency; two-sided percentile p;
Benjamini-Hochberg over all frequency and time bins of the region at 0.05.

Every spectral claim is decided on the events of reading (b1), the main reading, and shown on
the variant (b3) and, when it keeps other events than (b1), the variant (b1p) (see "Event
sets"). (b2) leaves no events in three subjects, so the group spectrograms over 12 subjects and
the per-subject tests cannot be formed under it; it is not used here.

In [ ]:
# (b1) decides; the variants (b3) and, when it holds other events, (b1p) are shown beside it
SETS = ["set_b1", "set_b3"] + (["set_b1p"] if (ev.set_b1p != ev.set_b1).any() else [])
print("event sets of the spectral claims:", SETS, "; (b1p) differs from (b1) in", int((ev.set_b1p != ev.set_b1).sum()), "events")
roi_ev = np.concatenate([spec[p]["roi"] for p in A.path])          # events, regions, freqs, bins
bmask = (TB >= -1.5 - 1e-9) & (TB <= -1 + 1e-9)
win = (TB >= 0 - 1e-9) & (TB <= 0.5 + 1e-9)
lo_f = (F >= 4) & (F <= 30)
hi_f = (F >= 50) & (F <= 120)
lf_i, hf_i = (F >= 8) & (F <= 32), (F >= 76) & (F <= 100)


def bh(p, q=0.05):
    p = np.asarray(p).ravel()
    o = np.argsort(p)
    th = q * np.arange(1, len(p) + 1) / len(p)
    ok = p[o] <= th
    k = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
    sig = np.zeros(len(p), bool)
    sig[o[:k]] = True
    return sig


def group_spectrogram(setname, seed):
    sel_ev = ev[setname].to_numpy()
    subj_med = {s: np.nanmedian(roi_ev[sel_ev & (ev.subject == s).to_numpy()], 0) for s in SUBJECTS}
    grp = np.mean([subj_med[s] for s in SUBJECTS], 0)
    grp = grp - np.median(grp[:, :, bmask], axis=2, keepdims=True)
    rng_mask = np.random.default_rng(seed)
    S = np.stack([subj_med[s] for s in SUBJECTS])                    # subjects, regions, freqs, bins
    nb = int(bmask.sum())
    reps = int(np.ceil(2000 / nb))
    masked = np.zeros_like(grp)
    nsig = {}
    for r in range(len(ajile.REGIONS)):
        pvals = np.zeros(grp.shape[1:])
        for fi in range(len(F)):
            base = S[:, r, fi][:, bmask]                               # subjects x baseline bins
            null = np.concatenate([np.median(np.stack([rng_mask.permutation(b) for b in base]), 0) for _ in range(reps)])
            obs = grp[r, fi]
            hi = (null[None, :] >= obs[:, None]).mean(1)
            lo = (null[None, :] <= obs[:, None]).mean(1)
            pvals[fi] = np.minimum(1, 2 * np.minimum(hi, lo))
        sig = bh(pvals).reshape(pvals.shape)
        masked[r] = np.where(sig, grp[r], 0)
        nsig[ajile.REGIONS[r]] = int(sig.sum())
    return dict(subj_med=subj_med, grp=grp, masked=masked, nsig=nsig, nnull=reps * nb)


G = {SET: group_spectrogram(SET, seed=3) for SET in SETS}
for SET in SETS:
    g = G[SET]
    print(f"{SET}: null values per frequency {g['nnull']}; significant bins per region (of {g['grp'].shape[1] * g['grp'].shape[2]}):", g["nsig"])
    print(f"   group spectrogram range (dB): {g['grp'].min():.2f} to {g['grp'].max():.2f}; time bins {TB[0]} to {TB[-1]} s")

> fig3-group-lfb-decrease-hfb-increase: At the group level, low-frequency power decreases and
> high-frequency power increases at movement initiation in several cortical regions.

In [ ]:
t0 = int(np.argmin(np.abs(TB)))
three = ["precentral", "postcentral", "inferior parietal"]
F3 = {}
for SET in SETS:
    masked, grp = G[SET]["masked"], G[SET]["grp"]
    rows = []
    for r, name in enumerate(ajile.REGIONS):
        rows.append(dict(region=ajile.REGION_NAMES[name],
                         a_low=masked[r][lo_f][:, win].mean(), a_high=masked[r][hi_f][:, win].mean(),
                         b_low=masked[r][lo_f, t0].mean(), b_high=masked[r][hi_f, t0].mean(),
                         unmasked_a_low=grp[r][lo_f][:, win].mean(), unmasked_a_high=grp[r][hi_f][:, win].mean()))
    f3 = pd.DataFrame(rows)
    f3["a_holds"] = (f3.a_low < 0) & (f3.a_high > 0)
    f3["b_holds"] = (f3.b_low < 0) & (f3.b_high > 0)
    F3[SET] = f3
    print(f"events {SET}")
    print(f3.round(3).to_string(index=False))
    for rd in ("a", "b"):
        print(f"  reading {rd}: holds in {int(f3[f3.region.isin(three)][rd + '_holds'].sum())} of the 3 named regions; in {int(f3[rd + '_holds'].sum())} of 8")
masked = G["set_b1"]["masked"]
fig, axs = plt.subplots(2, 4, figsize=(14, 6), sharex=True, sharey=True)
for r, ax in enumerate(axs.ravel()):
    im = ax.pcolormesh(TB, F, masked[r], cmap="RdBu_r", vmin=-1.5, vmax=1.5, shading="nearest")
    ax.set_title(ajile.REGION_NAMES[ajile.REGIONS[r]])
fig.colorbar(im, ax=axs, label="dB (masked), events b1")
fig.supxlabel("time from onset (s)")
fig.supylabel("frequency (Hz)")
plt.show()
print(f"plotted (b1) masked values range {masked.min():.2f} to {masked.max():.2f} dB; values outside the colour scale +-1.5 dB: {int((np.abs(masked) > 1.5).sum())}")

**Result: reproduced.** In precentral, postcentral and inferior parietal regions the masked 4-30
Hz mean is negative and the masked 50-120 Hz mean positive, both over 0 to 0.5 s (reading a;
precentral -0.871 and 0.480 dB under (b1)) and at the bin nearest onset (reading b), under (b1)
and (b3). Under (b1) the pattern holds in 7 of 8 regions by reading (a) and 4 of 8 by reading
(b).

> fig3-sensorimotor-largest: Spectral power changes are largest in precentral, postcentral and
> inferior parietal regions, and the three temporal regions show minimal movement-related
> changes.

In [ ]:
w2 = (TB >= -0.5 - 1e-9) & (TB <= 0.5 + 1e-9)
temporal = ["superior temporal", "middle temporal", "inferior temporal"]
for SET in SETS:
    masked = G[SET]["masked"]
    rows = []
    for r, name in enumerate(ajile.REGIONS):
        m = np.abs(masked[r])
        rows.append(dict(region=ajile.REGION_NAMES[name], a=m[:, win].mean(), b_low=m[lo_f][:, win].mean(),
                         b_high=m[hi_f][:, win].mean(), c=m[:, w2].mean()))
    f3b = pd.DataFrame(rows).set_index("region")
    print(f"events {SET}")
    print(f3b.round(4).to_string())
    for col in ("a", "b_low", "b_high", "c"):
        top3 = list(f3b[col].sort_values(ascending=False).index[:3])
        t_below = all(f3b.loc[t, col] < f3b.loc[three, col].min() for t in temporal)
        print(f"  {col}: top three {top3}; named three are the top three: {set(top3) == set(three)}; "
              f"named in top three: {len(set(top3) & set(three))}; every temporal region below all three: {t_below}")
    rank_b = (f3b.b_low.rank(ascending=False) <= 3) & (f3b.b_high.rank(ascending=False) <= 3)
    print("  reading b, regions in the top three in both bands:", list(f3b.index[rank_b]))

**Result: reproduced.** Under all three readings, and under (b1) and (b3), precentral,
postcentral and inferior parietal are the three regions with the largest mean absolute masked
power, and every temporal region is below all three (reading a under (b1): 0.502, 0.495 and
0.325 dB against 0.045, 0.003 and 0.018 for the temporal regions).

> fig3-lfb-more-widespread: Low-frequency power decreases are more spatially widespread than
> high-frequency increases and are also present in supramarginal and superior temporal regions.

In [ ]:
for SET in SETS:
    masked = G[SET]["masked"]
    neg = {ajile.REGION_NAMES[n]: int((masked[r][lo_f][:, win] < 0).sum()) for r, n in enumerate(ajile.REGIONS)}
    pos = {ajile.REGION_NAMES[n]: int((masked[r][hi_f][:, win] > 0).sum()) for r, n in enumerate(ajile.REGIONS)}
    print(f"events {SET}")
    print("  significant negative 4-30 Hz bins over 0-0.5 s per region:", neg)
    print("  significant positive 50-120 Hz bins over 0-0.5 s per region:", pos)
    ra_low, ra_high = sum(v > 0 for v in neg.values()), sum(v > 0 for v in pos.values())
    rb_low, rb_high = sum(neg.values()) / lo_f.sum(), sum(pos.values()) / hi_f.sum()
    print(f"  reading a: regions with a negative LFB bin {ra_low}, with a positive HFB bin {ra_high}")
    print(f"  reading b: negative LFB bins per frequency bin {rb_low:.2f} ({lo_f.sum()} bins), positive HFB bins per frequency bin {rb_high:.2f} ({hi_f.sum()} bins)")
    print("  supramarginal and superior temporal negative LFB bins:", neg["supramarginal"], neg["superior temporal"])

**Result: reproduced.** Under (b1), 8 regions have a significant LFB decrease and 7 a
significant HFB increase over 0 to 0.5 s (reading a), and there are 36.2 significant negative
bins per LFB frequency bin against 23.4 positive bins per HFB frequency bin (reading b);
supramarginal and superior temporal have 30 and 25 significant negative LFB bins. (b3) gives 8
against 7 and 35.2 against 23.9.

> fig4-precentral-subject-variability: In the precentral region every subject shows the
> low/high-frequency pattern, but amplitudes and frequency bands vary; subjects 03, 06, 07, 08 and
> 11 have increases up to 120 Hz, subjects 09 and 12 mainly at 60-80 Hz, and subjects 04 and 08
> have narrower low-frequency decreases.

In [ ]:
for SET in SETS:
    spec_s = {}
    for s in SUBJECTS:
        m = G[SET]["subj_med"][s][pre]
        m = m - m[:, bmask].mean(1, keepdims=True)
        spec_s[s] = m[:, win].mean(1)
    SP = pd.DataFrame(spec_s, index=F).T
    print(f"events {SET}: precentral spectrum per subject, 0-0.5 s (dB):")
    print(SP.round(2).to_string())
    p1 = {s: bool((SP.loc[s, lf_i].mean() < 0) and (SP.loc[s, hf_i].mean() > 0)) for s in SUBJECTS}
    print("  part 1, LFB mean < 0 and HFB mean > 0:", p1, "->", all(p1.values()))
    p2 = {s: bool((SP.loc[s, [116, 121]] > 0).all()) for s in (3, 6, 7, 8, 11) if s in SUBJECTS}
    print("  part 2, 116 and 121 Hz positive:", p2, "->", all(p2.values()))
    p3 = {}
    for s in [s for s in (9, 12) if s in SUBJECTS]:
        hi = SP.loc[s, F >= 56]
        fmax = hi.idxmax()
        p3[s] = bool((61 <= fmax <= 76) and (SP.loc[s, (F >= 101) & (F <= 121)].mean() < hi.max() / 2))
        print(f"     S{s:02d}: largest 56-121 Hz bin at {fmax} Hz ({hi.max():.2f} dB); mean 101-121 Hz {SP.loc[s, (F >= 101) & (F <= 121)].mean():.2f} dB")
    print("  part 3:", p3, "->", all(p3.values()))
    width = {s: int((SP.loc[s, F <= 46] < SP.loc[s, F <= 46].min() / 2).sum()) for s in SUBJECTS}
    third = sorted(width.values())[min(2, len(width) - 1)]
    print("  part 4, width of the decrease (1-46 Hz bins below half of the most negative):", width)
    print("     three fewest (ties broken by subject order):", sorted(width, key=lambda s: width[s])[:3],
          "; subjects with width <= the third smallest:", [s for s in SUBJECTS if width[s] <= third],
          "; S04 and S08 at or below it:", [width.get(s, 99) <= third for s in (4, 8)])

**Result: partially reproduced.** Parts 2 and 3 hold: the 116 and 121 Hz bins are positive in
S03, S06, S07, S08 and S11, and the largest 56-121 Hz bin is at 71 Hz in S09 and 66 Hz in S12,
with the 101-121 Hz mean below half of it. Part 4 holds: the widths of the decrease are 1 (S08)
and 2 (S04) bins under (b1), the two smallest; under (b3) S08 has 2 and S04 shares the third
smallest width (3) with S02, S03, S05 and S07. Part 1 fails in S09, whose 76-100 Hz mean is not
positive, under both event sets.

### Banded precentral power per event

Band power from the region spectrogram of each event over 0 to 0.5 s: LFB bins 11 to 31 Hz,
HFB bins 76 to 96 Hz, the mean over bins (Methods; main) and the median over time bins then
over frequency bins (code; variant). Without baseline: the projection of each electrode's
10*log10 band mean with no subtraction.

In [ ]:
pr = roi_ev[:, pre][:, :, win]                                        # events, freqs, bins
bp_all = pd.DataFrame({"lfb_mean": pr[:, lf_i].mean((1, 2)), "hfb_mean": pr[:, hf_i].mean((1, 2)),
                       "lfb_median": np.median(np.median(pr[:, lf_i], 2), 1), "hfb_median": np.median(np.median(pr[:, hf_i], 2), 1)})
raw_l, raw_h, bl_l = [], [], []
for p in A.path:
    w = proj[p][:, pre]
    b = spec[p]["band"]
    raw_l.append(b[:, :, 4] @ w)
    raw_h.append(b[:, :, 6] @ w)
    bl_l.append(b[:, :, 0] @ w)
bp_all["lfb_raw"], bp_all["hfb_raw"] = np.concatenate(raw_l), np.concatenate(raw_h)
bp_all["lfb_check"] = np.concatenate(bl_l)
bp_all[["subject", "day"]] = ev[["subject", "day"]].to_numpy()
BP = {SET: bp_all[ev[SET].to_numpy()].reset_index(drop=True) for SET in SETS}
print("check, band mean from the region spectrogram equals the projection of electrode band means:",
      float(np.nanmax(np.abs(BP["set_b1"].lfb_mean - BP["set_b1"].lfb_check))))
for SET in SETS:
    print(f"events {SET}:")
    print(BP[SET].drop(columns=["lfb_check"]).describe().T[["count", "mean", "min", "max"]].round(2).to_string())

> fig5-band-power-differs-across-subjects: Precentral LFB and HFB power in the 0.5 s after
> movement onset differs significantly across subjects.

In [ ]:
for SET in SETS:
    bp = BP[SET]
    for c in ("lfb_mean", "hfb_mean", "lfb_median", "hfb_median"):
        h, p = kruskal([g[c].to_numpy() for _, g in bp.groupby("subject")])
        print(f"{SET} {c}: Kruskal-Wallis H = {h:.1f}, p = {p:.2e} ({bp.subject.nunique()} subjects, {len(bp)} events)")

**Result: reproduced.** Kruskal-Wallis across the 12 subjects gives p below 1e-300 (printed as
0) for LFB (H = 762.0) and HFB (H = 1244.4) under (b1), and under (b3) (H = 773.9 and 1253.1),
with the band median as well.

> table1-subject-band-power-ci: Each subject's precentral band power lies in the 95% bootstrap
> interval Table 1 gives; LFB power is below baseline and HFB power above it for most subjects.

In [ ]:
T1 = {1: (-1.87, -1.29, 0.03, 0.23), 2: (-1.12, -0.31, 0.07, 0.64), 3: (-2.11, -1.73, 1.39, 1.67), 4: (-0.17, 0.01, 0.32, 0.52),
      5: (-0.7, -0.38, 0.2, 0.48), 6: (-1.63, -1.11, 2.02, 2.37), 7: (-0.84, -0.51, 1.14, 1.34), 8: (-0.3, 0.04, 0.36, 0.63),
      9: (-0.57, -0.08, -0.11, 0.23), 10: (-0.87, -0.35, 0.01, 0.3), 11: (-2.3, -1.86, 0.09, 0.34), 12: (-0.66, -0.44, 0.06, 0.2)}
for SET in SETS:
    bp = BP[SET]
    rng_t1 = np.random.default_rng(1)
    rows = []
    for s in SUBJECTS:
        g = bp[bp.subject == s]
        for band, (lo_i, hi_i) in (("LFB", (0, 1)), ("HFB", (2, 3))):
            lo, hi = T1[s][lo_i], T1[s][hi_i]
            for bandstat in ("mean", "median"):
                v = g[f"{band.lower()}_{bandstat}"].to_numpy()
                for evstat, fn in (("mean", np.mean), ("median", np.median)):
                    est = fn(v)
                    boot = fn(v[rng_t1.integers(0, len(v), (5000, len(v)))], axis=1)
                    rows.append(dict(subject=s, band=band, band_stat=bandstat, event_stat=evstat, estimate=est,
                                     paper_lo=lo, paper_hi=hi, inside=lo <= est <= hi,
                                     boot_lo=np.percentile(boot, 2.5), boot_hi=np.percentile(boot, 97.5)))
    t1 = pd.DataFrame(rows)
    summary = t1.groupby(["band_stat", "event_stat"]).inside.sum()
    print(f"events {SET}: subject and band pairs inside the paper's interval, of {2 * len(SUBJECTS)}:")
    print(summary.to_string())
    best = summary.idxmax()
    print("  best reading (band statistic, event statistic):", best, int(summary.max()))
    print(t1[(t1.band_stat == best[0]) & (t1.event_stat == best[1])].round(2).to_string(index=False))
    if SET == "set_b1":
        print("  the band mean and event mean reading, for comparison:")
        print(t1[(t1.band_stat == "mean") & (t1.event_stat == "mean")].round(2).to_string(index=False))
    mm = t1[(t1.band_stat == best[0]) & (t1.event_stat == best[1])]
    print(f"  under the best reading, subjects with LFB below 0: {int((mm[mm.band == 'LFB'].estimate < 0).sum())}; "
          f"with HFB above 0: {int((mm[mm.band == 'HFB'].estimate > 0).sum())} (of {len(SUBJECTS)})")

**Result: partially reproduced.** No reading puts all 24 subject and band values inside the
paper's intervals. The best reading is 21 of 24 under (b1) (band mean, median over events) and
22 of 24 under (b3) (band mean, mean over events); both are at least 18. Under (b1) the band
mean averaged over events, the Methods' reading, gives 8 of 24, pulled by events with zeroed
ECoG (event values from -311 to +317 dB). LFB is below 0 in 12 of 12 subjects and HFB above 0 in 12 (b1, best
reading) and 11 (b3) of 12.

> fig5-recording-day-effects: Precentral band power differs significantly across recording days
> in LFB for subjects 03, 05, 06, 07, 08 and 11, and in HFB for subjects 03, 04, 07, 08 and 10.

In [ ]:
paper_sig = {("LFB", s) for s in (3, 5, 6, 7, 8, 11)} | {("HFB", s) for s in (3, 4, 7, 8, 10)}
DY = {}
for SET in SETS:
    bp = BP[SET]
    rows = []
    for s in SUBJECTS:
        g = bp[bp.subject == s]
        for band in ("LFB", "HFB"):
            for st in ("mean", "median"):
                c = f"{band.lower()}_{st}"
                h, p = kruskal([x[c].to_numpy() for _, x in g.groupby("day")])
                rows.append(dict(subject=s, band=band, band_stat=st, p=p, sig=p < 0.05, paper_sig=(band, s) in paper_sig))
    dy = pd.DataFrame(rows)
    dy["agree"] = dy.sig == dy.paper_sig
    DY[SET] = dy
    print(f"events {SET}: Kruskal-Wallis p across days")
    print(dy.pivot_table(index="subject", columns=["band", "band_stat"], values="p").to_string(float_format=lambda x: f"{x:.3g}"))
    print(f"  pairs agreeing with the paper's pattern, of {2 * len(SUBJECTS)}:", dy.groupby("band_stat").agree.sum().to_dict())
    print("  disagreeing pairs (band mean):", dy[(dy.band_stat == "mean") & ~dy.agree][["subject", "band", "p", "paper_sig"]].round(4).values.tolist())
    print("  significant pairs (band mean):", int(dy[dy.band_stat == "mean"].sig.sum()))
    print("  per-day means (dB), band mean:")
    print(bp.groupby(["subject", "day"])[["lfb_mean", "hfb_mean"]].mean().unstack().round(2).to_string())

**Result: partially reproduced.** 23 of the 24 subject and band pairs agree with the paper's
pattern of significant day effects, under (b1) and (b3) and with the band mean or median. The
pair that differs is S10 HFB, significant in the paper (p = 0.006) and not here (p = 0.452 under
(b1), 0.200 under (b3)).

> extfig5-1-baseline-reduces-day-variability: Subtracting the -1.5 to -1 s baseline reduces the
> variability of band power across recording days, though clear variability remains.

Reading (a): precentral region, per subject, the SD across days of the day means of event band
power, with baseline (band mean) and without (projected 10*log10 band mean). Reading (b): the
same per electrode, on the days the electrode is marked good, for electrodes good on at least two
days.

In [ ]:
for SET in SETS:
    bp = BP[SET]
    rows = []
    for s in SUBJECTS:
        g = bp[bp.subject == s]
        dm = g.groupby("day")[["lfb_mean", "hfb_mean", "lfb_raw", "hfb_raw"]].mean()
        rows.append(dict(subject=s, lfb_sd_bl=dm.lfb_mean.std(), lfb_sd_raw=dm.lfb_raw.std(),
                         hfb_sd_bl=dm.hfb_mean.std(), hfb_sd_raw=dm.hfb_raw.std()))
    ra = pd.DataFrame(rows)
    print(f"events {SET}")
    print(ra.round(3).to_string(index=False))
    for b in ("lfb", "hfb"):
        print(f"  reading a, {b}: SD with baseline below SD without in {int((ra[b + '_sd_bl'] < ra[b + '_sd_raw']).sum())} of {len(ra)} subjects")
    el_rows = []
    for s in SUBJECTS:
        paths = list(A.path[A.subject == s])
        nch = spec[paths[0]]["band"].shape[1]
        for c in range(nch):
            dm = {k: [] for k in ("lb", "lr", "hb", "hr")}
            for p in paths:
                if not meta[p]["electrodes"].good.iloc[c]:
                    continue
                msk = ev[ev.path == p][SET].to_numpy()
                b = spec[p]["band"][msk, c]
                for k, j in (("lb", 0), ("lr", 4), ("hb", 2), ("hr", 6)):
                    dm[k].append(np.nanmean(b[:, j]))
            if len(dm["lb"]) >= 2:
                el_rows.append(dict(subject=s, ch=c, **{k: np.std(v, ddof=1) for k, v in dm.items()}))
    rb = pd.DataFrame(el_rows)
    for b, (kb, kr) in (("lfb", ("lb", "lr")), ("hfb", ("hb", "hr"))):
        print(f"  reading b, {b}: SD with baseline below SD without in {int((rb[kb] < rb[kr]).sum())} of {len(rb)} electrodes "
              f"(median SD with {rb[kb].median():.3f}, without {rb[kr].median():.3f} dB)")
    print("  by subject, fraction of electrodes with lower SD after baseline (LFB, HFB):")
    print(rb.assign(l=rb.lb < rb.lr, h=rb.hb < rb.hr).groupby("subject")[["l", "h"]].mean().round(2).T.to_string())
    print("  significant day effects found in fig5-recording-day-effects (band mean):", int(DY[SET][DY[SET].band_stat == "mean"].sig.sum()))

**Result: reproduced.** The SD across days of the day means is lower with baseline subtraction
than without in 10 and 9 of 12 subjects (LFB, HFB) for the precentral region and in 1018 and 971
of 1144 electrodes under (b1); in 10 and 11 of 12 subjects and 1028 and 1006 electrodes under
(b3). Variability remains: fig5-recording-day-effects finds 10 significant day effects.

> discussion-day-changes-small: Day-to-day changes in baseline-subtracted spectral power are
> small (about 1-2 dB) compared with spectral power without baseline subtraction (about 10-50
> dB).

In [ ]:
for SET in SETS:
    bp = BP[SET]
    rows = []
    for s in SUBJECTS:
        g = bp[bp.subject == s]
        for band in ("lfb", "hfb"):
            dm = g.groupby("day")[f"{band}_mean"].mean()
            rows.append(dict(subject=s, band=band, max_dev=np.abs(dm - g[f"{band}_mean"].mean()).max(),
                             day_range=dm.max() - dm.min(), level_uV=g[f"{band}_raw"].mean(), level_V=g[f"{band}_raw"].mean() - 120))
    dc2 = pd.DataFrame(rows)
    print(f"events {SET}")
    print(dc2.round(2).to_string(index=False))
    low = bp[(bp.lfb_raw < 0) | (bp.hfb_raw < 0)]
    print(f"  events whose precentral band power without baseline is below 0 dB: {len(low)} of {len(bp)}, "
          f"by subject {low.subject.value_counts().sort_index().to_dict()}")
    med = bp.groupby("subject")[["lfb_raw", "hfb_raw"]].median()
    print(f"  level without baseline as the median over events instead (uV): {med.min().min():.1f} to {med.max().max():.1f} dB")
    print(f"  largest day-to-day change (deviation of a day mean from the subject mean): {dc2.max_dev.max():.2f} dB; all at most 2.2 dB: {bool((dc2.max_dev <= 2.2).all())}; "
          f"pairs above 2.2 dB: {dc2[dc2.max_dev > 2.2][['subject', 'band']].values.tolist()}")
    print(f"  largest range of day means: {dc2.day_range.max():.2f} dB")
    for u in ("level_uV", "level_V"):
        print(f"  level without baseline, {u}: {dc2[u].min():.1f} to {dc2[u].max():.1f} dB; all in 9-55 dB: {bool(dc2[u].between(9, 55).all())}")

**Result: blocked (methods underspecified).** Under (b3) every day mean is within 1.00 dB of its
subject's mean and the level without baseline is 12.3 to 36.4 dB in microvolts, so both parts
hold. Under (b1) the largest deviation is 8.15 dB (S10), above 2.2 dB in S02, S06 and S10, and
the level is 8.5 to 35.4 dB; both fail, because of 42 events whose precentral ECoG is zeroed in
the window. The two open readings of the segment rejection give different outcomes (question
asked). In volts every level is below 0 dB.

## Summary

| Claim | Outcome | Notebook |
|---|---|---|
| figshare-event-link (link) | reproduced | this one |
| figshare-roi-link (link) | reproduced | this one |
| subject-days (link) | reproduced | this one |
| cohort | partially reproduced | this one |
| fig2-event-counts | reproduced | this one |
| fig2-features-differ-across-subjects | reproduced | this one |
| fig2-reach-shape | partially reproduced | this one |
| fig2-bimanual-skew | blocked, methods underspecified | this one |
| extfig2-1c-contralateral-larger | reproduced | this one |
| extfig2-2-feature-correlations | reproduced | this one |
| fig2-days-and-waking | partially reproduced | this one |
| fig3-group-lfb-decrease-hfb-increase | reproduced | this one |
| fig3-sensorimotor-largest | reproduced | this one |
| fig3-lfb-more-widespread | reproduced | this one |
| fig4-precentral-subject-variability | partially reproduced | this one |
| fig5-band-power-differs-across-subjects | reproduced | this one |
| table1-subject-band-power-ci | partially reproduced | this one |
| fig5-recording-day-effects | partially reproduced | this one |
| extfig5-1-baseline-reduces-day-variability | reproduced | this one |
| discussion-day-changes-small | blocked, methods underspecified | this one |
| fig6-r2-at-most-025 | reproduced | regression |
| fig6b-best-models-sensorimotor | reproduced | regression |
| fig6c-angle-and-day-most-informative | reproduced | regression |
| extfig6-2-angle-most-retained | not reproduced | regression |
| fig6d-upward-reaches-enhance-pattern | reproduced | regression |
| fig6d-angle-and-day-largest-coefficients | reproduced | regression |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The false-positive flags are read from the paper's Figshare epoch files, linked to the
  dataset's events by their timestamps; the dataset does not mark the false positives.
- The regions use the paper's own projection matrices from Figshare, not a new projection from
  the AAL atlas, and the bad electrodes of each file are set to 0 before renormalising.
- The PSD rejection of segments is not specified beyond "power <0 dB" and "abnormally high
  power at 115-125 Hz (>3 SDs)"; three readings are run and a question is asked. Two claims
  are blocked because readings (b1) and (b3) give different outcomes.
- The ECoG of three files is read on a clock shifted by the session start (see the clock check).
- Kernel density, peak and tie rules of fig2-reach-shape and fig4 are standard choices stated
  in their cells.

What was found about the dataset:

- It holds the 12 subjects and 55 days of the paper and 8088 events, 1157 of which the paper's
  Figshare files mark as false positives; the dataset has no flag for them, and with them the
  per-subject counts do not match the paper's (211 to 996, median 770.5).
- In sub-01 day 7, sub-04 day 3 and sub-06 day 3 the ECoG starts at the session start while the
  pose and the events count from midnight, but the ECoG's starting time is stored as 0; on the
  stored clock 237 events lie beyond the end of the ECoG, and the movement response appears only
  on the shifted clock.
- Session start times are stored at offset -05:00, but the sleep labels fall at night only when
  they are read on the UTC clock.
- The pose is labelled in meters but holds pixels; the ECoG is labelled volts with conversion
  1e-6, consistent with values in microvolts.
- Bimanual overlap is described as a fraction of the movement duration but holds seconds
  (frames of the Figshare files divided by 30).
- The electrode groups whose description says "depth" are in 7 subjects, where the paper
  counts 5 subjects with depth electrodes.
- Some events have ECoG zeroed in the analysed window (42 of the events kept by reading b1);
  the files do not mark the zeroed stretches.
- Sex and age of the subjects are not in the files.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")